<a href="https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

## 1. Build the feature vector

This notebook audits the feature vector used with the anonymized 30,000-row starter dataset.

The W03 ML-04 data contract defines the main future-prediction framing as:

**March 2026 features → April 2026 outcome**

That contract uses five March features for the future-looking experiment.

This optional ML-05 notebook instead audits the feature vector used by the repository's starter-data modeling pipeline and the completed W04 baseline. The W04 baseline is a current-window decision-support ranking rule, not a prediction of future performance.

For consistency with the repository pipeline, I use the same model numeric and categorical feature groups defined in `scripts/ml_utils.py` and exclude identifiers, the target, label-derived fields, recent trend windows, and product-generation metadata.

The feature vector contains:

- numeric search, content, traffic, activity, and performance features
- log-transformed traffic features
- categorical context encoded as one-hot columns
- explicit handling of missing numeric and categorical values

The target is `is_declining_label`, but the target itself is never included in the final feature matrix.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Setup and repository-aligned feature definitions.

from pathlib import Path
import os
import numpy as np
import pandas as pd
from IPython.display import display

# ---------------------------------------------------------
# Resolve the repository.
# This works both in Colab and when running locally.
# ---------------------------------------------------------

REPO_DIR = Path("/content/flyrank-internship-ml")

if os.environ.get("COLAB_RELEASE_TAG"):
    if not REPO_DIR.exists():
        !git clone -q https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml {REPO_DIR}
    ROOT = REPO_DIR
else:
    candidates = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
    ]

    ROOT = next(
        (
            p for p in candidates
            if (p / "data/raw/content_refresh_anonymized.csv").exists()
        ),
        None,
    )

    if ROOT is None:
        raise FileNotFoundError(
            "Could not locate the repository containing "
            "data/raw/content_refresh_anonymized.csv."
        )

DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print(f"Repository: {ROOT}")
print(f"Dataset: {len(df):,} rows × {len(df.columns)} columns")


# ---------------------------------------------------------
# These lists mirror scripts/ml_utils.py.
# ---------------------------------------------------------

MODEL_NUMERIC_FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

MODEL_CATEGORICAL_FEATURES = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]


# Fields that must not become model inputs.
EXCLUDED_FIELDS = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "provider_used",
    "model_used",
    "is_declining_label",
]

print("\nModel numeric features:", len(MODEL_NUMERIC_FEATURES))
print("Model categorical features:", len(MODEL_CATEGORICAL_FEATURES))
print("Excluded fields:", len(EXCLUDED_FIELDS))

Repository: /content/flyrank-internship-ml
Dataset: 30,000 rows × 44 columns

Model numeric features: 18
Model categorical features: 8
Excluded fields: 13


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the actual feature vector.

feature_df = df.copy()

# ---------------------------------------------------------
# Engineer the same log features used by the repository.
# ---------------------------------------------------------

feature_df["log_impressions_90d"] = np.log1p(
    pd.to_numeric(feature_df["impressions_90d"], errors="coerce")
)

feature_df["log_clicks_90d"] = np.log1p(
    pd.to_numeric(feature_df["clicks_90d"], errors="coerce")
)

feature_df["log_sessions_90d"] = np.log1p(
    pd.to_numeric(feature_df["sessions_90d"], errors="coerce")
)

feature_df["log_ai_sessions_90d"] = np.log1p(
    pd.to_numeric(feature_df["ai_sessions_90d"], errors="coerce")
)


# ---------------------------------------------------------
# Numeric features
# ---------------------------------------------------------

numeric_frame = feature_df[MODEL_NUMERIC_FEATURES].apply(
    pd.to_numeric,
    errors="coerce"
)

numeric_frame = (
    numeric_frame
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)


# ---------------------------------------------------------
# Categorical features
# ---------------------------------------------------------

categorical_frame = (
    feature_df[MODEL_CATEGORICAL_FEATURES]
    .fillna("unknown")
    .astype(str)
)

encoded_categorical = pd.get_dummies(
    categorical_frame,
    prefix=MODEL_CATEGORICAL_FEATURES,
    dummy_na=False,
    dtype=float,
)


# ---------------------------------------------------------
# Combine numeric + categorical features
# ---------------------------------------------------------

X = pd.concat(
    [
        numeric_frame.reset_index(drop=True),
        encoded_categorical.reset_index(drop=True),
    ],
    axis=1,
)


# Target is kept separately.
y = (
    feature_df["trend_direction"]
    .fillna("")
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)


print("FEATURE VECTOR")
print("==============")
print(f"Rows: {X.shape[0]:,}")
print(f"Encoded feature columns: {X.shape[1]:,}")
print(f"Target rows: {len(y):,}")

print("\nFirst 15 feature columns:")
print(X.columns[:15].tolist())

print("\nFeature matrix preview:")
display(X.head())

FEATURE VECTOR
Rows: 30,000
Encoded feature columns: 52
Target rows: 30,000

First 15 feature columns:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position']

Feature matrix preview:


,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,...,word_count_tier_unknown,impression_tier_excellent,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,0.0,88,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,10.0,0.00,0.00,0.0,0.0,9.371779,4.077537,4.369448,0.0,88,...,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.0,0.00,0.00,2803.0,17469.0,9.859588,3.218876,4.983607,0.0,88,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

## 2. Feature notes (meaning, missing, categorical, available-when?)

The feature vector follows the repository's existing modeling feature lists.

### Numeric features

Numeric features describe search context, content characteristics, historical traffic, activity, and observed performance.

The four traffic-count features receive `log1p` transformations:

- `impressions_90d`
- `clicks_90d`
- `sessions_90d`
- `ai_sessions_90d`

The log transformation reduces the effect of extremely large traffic values while preserving their ordering.

### Missing values

Numeric missing values are replaced with `0`, matching the repository's existing feature-preparation and modeling pipeline.

Categorical missing values are represented as `"unknown"` before one-hot encoding.

This is intentionally consistent with the reference pipeline rather than introducing a new preprocessing policy in this notebook.

### Categorical features

Categorical context includes:

- competition level
- content type
- main intent
- age tier
- freshness tier
- word-count tier
- impression tier
- position tier

These are converted into one-hot encoded columns.

### Availability

For the starter-data/W04 ranking context, these features describe information observed in the current 90-day window or existing content metadata.

For the W03 future-prediction contract, the stricter rule still applies: a feature must be known before the April outcome window. The W03 March-only five-feature frame remains the authoritative future-looking contract.

Therefore, this notebook audits feature construction and leakage rather than changing the W03 prediction definition.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect missingness and confirm that the feature vector
# contains only the intended model feature groups.

print("MISSINGNESS CHECK")
print("=================")

numeric_missing = (
    feature_df[MODEL_NUMERIC_FEATURES]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Numeric fields with missing values before filling:")
display(
    numeric_missing[numeric_missing > 0].to_frame("missing_rows")
)

categorical_missing = (
    feature_df[MODEL_CATEGORICAL_FEATURES]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nCategorical fields with missing values before filling:")
display(
    categorical_missing[categorical_missing > 0].to_frame("missing_rows")
)


print("\nFINAL FEATURE MATRIX CHECK")
print("==========================")

print(f"Rows: {len(X):,}")
print(f"Columns: {X.shape[1]:,}")
print(f"Missing values in X: {int(X.isna().sum().sum())}")
print(
    f"Infinite values in X: "
    f"{int(np.isinf(X.select_dtypes(include=[np.number])).sum().sum())}"
)

assert X.isna().sum().sum() == 0
assert np.isinf(
    X.select_dtypes(include=[np.number])
).sum().sum() == 0

print("\nAll feature-matrix values are finite.")

MISSINGNESS CHECK
Numeric fields with missing values before filling:


,missing_rows
char_count,7699
word_count,7699
competition,2468
search_volume,2468
cpc,2468
scroll_rate,125



Categorical fields with missing values before filling:


,missing_rows
word_count_tier,7699
competition_level,2610
main_intent,2374



FINAL FEATURE MATRIX CHECK
Rows: 30,000
Columns: 52
Missing values in X: 0
Infinite values in X: 0

All feature-matrix values are finite.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 3. The leakage hunt

I will attack the feature vector using three leakage checks.

### 1. Label-derived fields

The starter label is:

`is_declining_label = (trend_direction == "down")`

Therefore:

- `trend_direction` must never be a feature.
- `trend_pct` must never be a feature because it is the numeric trend used to construct the label.

I will deliberately add `trend_pct` to a test model and compare its performance with the honest feature vector.

### 2. Overlapping 30-day windows

The fields:

- `impressions_last_30d`
- `impressions_prev_30d`

and their click/session equivalents describe the same comparison used to create the trend label.

I will test whether the impression windows can reconstruct the target.

If they can reconstruct the label, they are leakage for this target.

### 3. Product or generation metadata

`provider_used` and `model_used` describe how content was generated. The repository's data dictionary marks them as non-model features.

They are therefore excluded from the model feature vector.

The W04 baseline also uses only current observed signals and does not use these fields as rule inputs.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Deliberately attack the feature vector for leakage.

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score


# ---------------------------------------------------------
# 1. Basic label information
# ---------------------------------------------------------

print("LABEL")
print("=====")

print(f"Positive rows (declining): {int(y.sum()):,}")
print(f"Negative rows: {int((1 - y).sum()):,}")
print(f"Declining rate: {y.mean():.1%}")


# ---------------------------------------------------------
# 2. Label-derived feature: trend_pct
# ---------------------------------------------------------

trend_pct = pd.to_numeric(
    feature_df["trend_pct"],
    errors="coerce"
).fillna(0)

trend_pct_proxy = (trend_pct < -20).astype(int)

trend_pct_mismatches = int(
    (trend_pct_proxy != y).sum()
)

print("\nLEAKAGE TEST 1 — trend_pct")
print("==========================")
print(
    f"Rows where trend_pct threshold disagrees with label: "
    f"{trend_pct_mismatches:,}"
)


# ---------------------------------------------------------
# 3. Reconstruct the label from the 30-day windows
# ---------------------------------------------------------

last_30 = pd.to_numeric(
    feature_df["impressions_last_30d"],
    errors="coerce"
).fillna(0)

previous_30 = pd.to_numeric(
    feature_df["impressions_prev_30d"],
    errors="coerce"
).fillna(0)

window_label = (
    (previous_30 > 0)
    & (last_30 < 0.80 * previous_30)
).astype(int)

window_mismatches = int(
    (window_label != y).sum()
)

print("\nLEAKAGE TEST 2 — 30-day windows")
print("===============================")
print(
    f"Rows where the 30-day window reconstruction "
    f"disagrees with the label: {window_mismatches:,}"
)


# ---------------------------------------------------------
# 4. Model with honest features
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

honest_model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42,
)

honest_model.fit(X_train, y_train)

honest_probability = honest_model.predict_proba(X_test)[:, 1]

honest_auc = roc_auc_score(
    y_test,
    honest_probability
)


# ---------------------------------------------------------
# 5. Deliberately add the leaked trend_pct
# ---------------------------------------------------------

X_leaky = X.copy()
X_leaky["trend_pct"] = trend_pct.values

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

leaky_model = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42,
)

leaky_model.fit(Xl_train, yl_train)

leaky_probability = leaky_model.predict_proba(Xl_test)[:, 1]

leaky_auc = roc_auc_score(
    yl_test,
    leaky_probability
)


# ---------------------------------------------------------
# Results
# ---------------------------------------------------------

print("\nMODEL LEAKAGE COMPARISON")
print("========================")
print(f"Honest AUC: {honest_auc:.3f}")
print(f"Leaky AUC:  {leaky_auc:.3f}")

print("\nInterpretation:")
print(
    "Adding trend_pct should cause a very large increase in AUC "
    "because trend_pct is derived from the same trend used to define the label."
)

print(
    "\nThe 30-day window reconstruction should have zero mismatches "
    "with the starter label."
)

LABEL
=====
Positive rows (declining): 16,262
Negative rows: 13,738
Declining rate: 54.2%

LEAKAGE TEST 1 — trend_pct
Rows where trend_pct threshold disagrees with label: 4

LEAKAGE TEST 2 — 30-day windows
Rows where the 30-day window reconstruction disagrees with the label: 0

MODEL LEAKAGE COMPARISON
Honest AUC: 0.663
Leaky AUC:  1.000

Interpretation:
Adding trend_pct should cause a very large increase in AUC because trend_pct is derived from the same trend used to define the label.

The 30-day window reconstruction should have zero mismatches with the starter label.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.